In [1]:
# importando os scripts de consulta

import sys
import os

# Altera o caminho para a pasta raiz do projeto (um nível acima)
raiz_projeto = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Adiciona a raiz do projeto ao caminho de busca do Python se ainda não estiver lá
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

    
import pandas as pd
from scripts.crm_leads import load_leads

In [2]:
captacao = load_leads()



Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 250 registros
Página 14: 250 registros
Página 15: 250 registros
Página 16: 250 registros
Página 17: 250 registros
Página 18: 250 registros
Página 19: 250 registros
Página 20: 250 registros
Página 21: 250 registros
Página 22: 250 registros
Página 23: 250 registros
Página 24: 250 registros
Página 25: 250 registros
Página 26: 250 registros
Página 27: 250 registros
Página 28: 250 registros
Página 29: 250 registros
Página 30: 250 registros
Página 31: 250 registros
Página 32: 145 registros


In [3]:

url = 'https://docs.google.com/spreadsheets/d/13DBRdUsHT5wupKkq3Omi9VUr9Nj3R5gYVMfSSkmtXQM/export?format=xlsx'

matriculas_itabirito = pd.read_excel(url, sheet_name = "ITABIRITO")
matriculas_altamira = pd.read_excel(url, sheet_name = "ALTAMIRA")
matriculas_lorena = pd.read_excel(url, sheet_name = "LORENA")
matriculas_canaa = pd.read_excel(url, sheet_name = "Canaã")


matriculas = pd.concat([matriculas_itabirito, matriculas_altamira, matriculas_lorena, matriculas_canaa], ignore_index=True)

In [4]:
matriculas.dropna(inplace = True, subset = ['CPF (preencher apenas com números)'])

In [5]:
captacao.columns

Index(['Tabulação Realizada', 'Nome Completo', 'Nome', 'Sobrenome', 'Email',
       'Celular', 'Telefone', 'ID Lead', 'Data Criação', 'Unidade Form',
       'Situação Funil', 'Origem Lead', 'Canal', 'cpf', 'Concurso Atual',
       'Concurso de Origem', 'Número Inscrição', 'Oferta Atual',
       'Linha de Negócio', 'Unidade Lead', 'Última Tabulação', 'Idade',
       'Data Nascimento', 'Proprietário', 'Nacionalidade', 'Estado Civil',
       'Órgão Expedidor', 'RG', 'Sexo', 'Cor/Raça', 'Tabulação Lead',
       'Detalhamento Tabulação'],
      dtype='str')

In [6]:
colunas_captacao = ['Nome Completo', 'cpf', 'ID Lead', 'Proprietário', 'Situação Funil']
captacao = captacao[colunas_captacao]

## Mapeando leads sem proprietário e buscando na planilha

In [7]:
# leads sem proprietário

matriculas_sem_proprietario = captacao[captacao['Proprietário'].isnull() &  (captacao['Situação Funil'] == 'Matriculado') ]

In [8]:
matriculas_sem_proprietario.shape

(43, 5)

In [9]:
df = pd.merge(matriculas_sem_proprietario, matriculas, how='left', left_on='Nome Completo', right_on='NOME')

In [10]:
df[['Nome Completo', 'cpf', 'ID Lead', 'Proprietário', 'NOME', 'CONSULTOR (a)', 'CPF (preencher apenas com números)']].loc[~df['CONSULTOR (a)'].isnull()].to_excel('matriculas_sem_proprietario.xlsx', index=False)

## Mapear matrículas que diverge o proprietário do consultor

In [11]:
matriculados = captacao[captacao['Situação Funil'] == 'Matriculado']
matriculados.shape

(760, 5)

In [12]:
df = matriculados.merge(matriculas,  how='left', left_on='Nome Completo', right_on='NOME')

In [13]:


proprietarios_consultores = {
    'Maria Eduarda Freire': 'Duda',
    'Maria Rafaela da Rocha Tavares': 'Maria Rafaela',
    'Adauri Marques Delfino Junior': 'Adauri',
    'Karen Flávia': 'Flávia',
    'Daniel Faria Santos': 'Daniel',
    'Geovana Souza da Silva': 'Geovana',
    'Agnata Pamela Silva Souza': 'Agnata',
    'Giovanna Eduarda Do Carmo Braga': 'Giovanna Braga',
    'Hailton Alves Júnior': 'Hailton',
    'Karolaine de Oliveira Alves Souza': 'Karolaine',
    'Thiago Oliveira da Cruz': 'Thiago',
    'Maria Fernanda Luiz Monteiro': 'Maria Fernanda ',
    'Ana Carolina Guimarães': 'Ana Carolina',
    'Lorena Viana Gonçalves Pereira': 'Lorena',
    'Thalia Gabriela de Araújo Silva': 'Thalia',
    'Camila Guimarães Santos': 'Camila',
    'Breno Assis Ferreira': 'Breno'
}

# verificando se o proprietário é diferente do consultor
df['Proprietário'] = df['Proprietário'].map(proprietarios_consultores)

# tirando linhas onde proprietário ou consultor são nulos
df = df.dropna(subset=['Proprietário', 'CONSULTOR (a)'])
divergencias = df[df['Proprietário'] != df['CONSULTOR (a)']][['Nome Completo', 'cpf', 'ID Lead', 'Proprietário', 'NOME', 'CONSULTOR (a)', 'CPF (preencher apenas com números)']]
display(divergencias.shape)


(3, 7)

In [14]:
divergencias.to_excel('divergencias_proprietario_consultor.xlsx', index=False)

Montando o relatório de matrículas por consultor no mês de julho